# Task 2
1、使用逻辑回归、SVM支持向量机对手写数字案例进行训练和预测

* 观察运行时间
* 观察准确率



2、对手写数字数据进行PCA降维，然后使用逻辑回归、SVM支持向量机对降维后的数据进行训练和预测

* 观察运行时间
* 观察准确率

## 导入包

In [58]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pydantic_settings import (
    BaseSettings, 
    SettingsConfigDict
)
import asyncio
from functools import partial
from typing import Callable, Iterable, Optional, Sequence, TypedDict,Union,List,Dict,Literal,Annotated
from enum import Enum,StrEnum,unique
from abc import ABC, abstractmethod


import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from loguru import logger
from pathlib import Path
from joblib import Parallel,delayed,Memory,dump,load

from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline,Parallel
from sklearn.svm import SVC

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langgraph.prebuilt import ToolNode

from langgraph.graph import END, START, StateGraph,MessagesState
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


sns.set_theme(style="whitegrid")

# Choose available CJK fonts first to avoid findfont / missing glyph warnings.
preferred_cjk_fonts = [
    "Noto Sans CJK SC",
    "Noto Sans CJK TC",
    "Noto Sans CJK JP",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
]
installed_fonts = {font.name for font in fm.fontManager.ttflist}
available_fonts = [name for name in preferred_cjk_fonts if name in installed_fonts]
plt.rcParams["font.sans-serif"] = available_fonts + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [44]:
class Settings(BaseSettings):
    data_path: Optional[Path] = Path("./data/inputs/digits.csv")
    openai_api_key: Optional[str]=None
    openai_base_url: Optional[str]=None
    openai_model: Optional[str]=None
    temperature: float = 0.0
    request_timeout: int = 30
    max_retries: int = 2
    enable_thinking: bool = True

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=False,
    )


cfg = Settings()

## 载入数据

In [27]:
class Data:
    """A class to handle data preprocessing and splitting.
    
    This class encapsulates the data loading, feature separation, and 
    train-test split functionality.
    """
    
    def __init__(self,file_path:Optional[Path|str]) -> None:
        """Initialize the Data object with raw data.
        
        Args:
            raw_data: A pandas DataFrame containing the raw data with a 'label' column.
        """
        self._data = self._load_data(file_path)
        self._X = self._data.drop(columns=["label"])
        self._y = self._data["label"]
    
    @property
    def X(self) -> pd.DataFrame:
        """Get the feature data."""
        return self._X
    
    @property
    def y(self) -> pd.Series:
        """Get the target labels."""
        return self._y
    
    def split_data(self, test_size: float = 0.2, random_state: int = 42) -> None:
        """Split the data into training and testing sets.
        
        Args:
            test_size: Proportion of dataset to include in the test split.
            random_state: Controls the shuffling applied to the data before applying the split.
        """
        # Split the data using stratified sampling to maintain label distribution
        (
            self._X_train,
            self._X_test,
            self._y_train,
            self._y_test
        ) = train_test_split(
            self._X,
            self._y,
            test_size=test_size,
            random_state=random_state,
            stratify=self._y
        )
    
    @property
    def X_train(self) -> pd.DataFrame:
        """Get the training features."""
        return self._X_train
    
    @property
    def X_test(self) -> pd.DataFrame:
        """Get the test features."""
        return self._X_test
    
    @property
    def y_train(self) -> pd.Series:
        """Get the training labels."""
        return self._y_train
    
    @property
    def y_test(self) -> pd.Series:
        """Get the test labels."""
        return self._y_test
    def get_info(self)-> Optional[pd.DataFrame]:
        """获取数据信息

        Returns:
            object: pd
        """
        return self._data.info()
    def get_describe(self)-> Optional[pd.DataFrame]:
        """获取数据描述性分析

        Returns:
            object: pd
        """
        return self._data.describe()
    def get_head(self,n:int=5)-> Optional[pd.DataFrame]:
        """获取数据前几行信息

        Returns:
            object: pd
        """
        return self._data.head(n)
    
    def _load_data(self,data_path: Optional[Union[Path, str]]) -> pd.DataFrame:
        """
        从CSV文件加载数据

        Args:
            data_path: CSV文件路径，可以是Path对象或字符串
            
        Returns:
            pandas.DataFrame: 加载的数据框
            
        Raises:
            FileNotFoundError: 当文件不存在时
            pd.errors.EmptyDataError: 当文件为空时
            pd.errors.ParserError: 当CSV解析失败时
            UnicodeDecodeError: 当编码错误时
            ValueError: 当输入参数无效时
        """
        # 输入验证
        if data_path is None:
            raise ValueError("data_path cannot be None")
        
        # 转换为Path对象以便统一处理
        path = Path(data_path)
        
        # 检查文件是否存在
        if not path.exists():
            raise FileNotFoundError(f"Data file not found: {path}")
        
        # 检查是否为文件（而不是目录）
        if not path.is_file():
            raise ValueError(f"Path is not a file: {path}")
        
        try:
            # 尝试使用UTF-8编码加载文件
            df = pd.read_csv(path, encoding='utf-8')
            logger.info(f"Successfully loaded data from {path}")
            return df
            
        except UnicodeDecodeError:
            # 如果UTF-8失败，尝试其他常见编码
            logger.warning(f"Failed to decode {path} with UTF-8, trying other encodings...")
            try:
                df = pd.read_csv(path, encoding='latin-1')
                logger.info(f"Loaded data from {path} using latin-1 encoding")
                return df
            except UnicodeDecodeError:
                # 最后尝试系统默认编码
                logger.warning(f"Failed to decode {path} with latin-1, trying system default...")
                df = pd.read_csv(path)
                logger.info(f"Loaded data from {path} using system default encoding")
                return df
                
        except Exception as e:
            # 记录详细错误信息
            logger.error(f"Error loading data from {path}: {str(e)}")
            raise
data=Data(cfg.data_path)
data.split_data()

2026-04-24 20:23:26.634 | INFO     | __main__:_load_data:125 - Successfully loaded data from data/inputs/digits.csv


## 建模

### 非降维

In [28]:
SolverType = Literal["lbfgs", "newton-cg", "newton-cholesky", "sag", "saga"]

In [29]:
def logistic_regression(
    data:Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[SolverType],
    random_state:int=42,
    max_iter:int=5000
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Perform logistic regression with hyperparameter tuning using cross-validation.
    
    Args:
        Cs: Regularization strength values to try
        cv: Number of cross-validation folds
        solvers: List of solvers to try
        
    Returns:
        Dictionary containing the best parameters and performance metrics,
        or None if no valid configuration was found
    """
    # Initialize best score and parameters
    best_score = float('-inf')
    best_params = {
        "solver": None,
        "score": best_score,
        "C": None,
        "coef": None,
        "intercept": None
    }
    
    # Validate inputs
    if not Cs or not solvers:
        return None
    
    # Use a more robust approach for handling potential errors
    for solver in solvers:
        try:
            # Create logistic regression model with cross-validation
            lr = LogisticRegressionCV(
                Cs=Cs, 
                cv=cv, 
                solver=solver,
                random_state=42,  
                max_iter=max_iter,
                scoring='accuracy' 
            )
            
            # Fit the model
            lr.fit(data.X_train, data.y_train)
            
            # Calculate test score
            current_score = lr.score(data.X_test, data.y_test)
            
            # Update best parameters if current is better
            if current_score > best_score:
                best_score = current_score
                best_params["C"] = lr.C_[0] if hasattr(lr.C_, '__len__') else lr.C_
                best_params["coef"] = lr.coef_.copy() if lr.coef_ is not None else None
                best_params["intercept"] = lr.intercept_[0] if hasattr(lr.intercept_, '__len__') else lr.intercept_
                best_params["score"] = best_score
                best_params["solver"] = solver
                
        except Exception as e:
            # Log warning instead of crashing
            logger.exception(f"Failed to evaluate solver {solver}: {str(e)}")
            continue
    
    # Return best parameters or None if no valid solution found
    return best_params if best_score != float('-inf') else None

In [54]:
def gridcv_svm(
    data: Data,
    Cs: Sequence[float],
    kernel: str | Sequence[str] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    **args
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Improved version with better error handling and parameter validation.
    
    Args:
        data: Training data containing X_train and y_train
        Cs: List of C values to try (must be positive)
        kernel: Kernel type(s) to try (should be in ['linear', 'poly', 'rbf', 'sigmoid'])
        gammas: Gamma values to try or 'auto' string
        cv: Number of cross-validation folds (positive integer)
    
    Returns:
        Dictionary with best parameters and results, or None if no valid result
    """
    # Input validation
    if not Cs:
        raise ValueError("Cs sequence cannot be empty")
    
    if cv is not None and cv <= 0:
        raise ValueError("cv must be a positive integer")
    
    # Validate that C values are positive
    if any(c <= 0 for c in Cs):
        raise ValueError("All C values must be positive")
    
    # Initialize best score
    best_score = float("-inf")
    
    # Handle kernel parameter
    kernel_values = [kernel] if isinstance(kernel, str) else list(kernel)
    
    # Handle gamma parameter
    gamma_values = [gammas] if isinstance(gammas, str) else list(gammas)
    
    # Convert Cs to list
    c_values = list(Cs)
    
    # Create parameter grid
    param_grid = {
        "C": c_values,
        "kernel": kernel_values,
        "gamma": gamma_values
    }
    
    # Create grid search model with improved settings
    model = GridSearchCV(
        SVC(random_state=42),  # Set random state for reproducibility
        param_grid=param_grid,
        cv=cv,
        n_jobs=-1,  # Parallel processing for speed
        verbose=0,
        scoring='accuracy'  # Explicitly specify scoring metric
    )
    
    try:
        # Fit the model
        model.fit(data.X_train, data.y_train)
        
        # Check if we found a valid solution
        if model.best_score_ == float("-inf"):
            return None
            
        # Extract results
        best_params = model.best_params_.copy()
        best_score = model.best_score_
        
        # Add additional information
        best_params["score"] = best_score
        best_params["support_vectors"] = model.best_estimator_.support_vectors_
        
        return best_params
        
    except Exception as e:
        # Log error but don't crash the application
        logger.exception(f"Error during grid search: {e}")
        return None

### PCA降维之后建模

In [51]:

def pca_logistic_regression(
    data: Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[SolverType],
    random_state: int = 42,
    max_iter: int = 5000,
    n_components: Union[float, int, str] = 0.95,
    **args
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Perform logistic regression with PCA using accuracy as the performance metric.
    """
    best_score = float("-inf")
    best_params = {
        "solver": None,
        "score": best_score,
        "C": None,
        "coef": None,
        "intercept": None
    }

    if not Cs or not solvers:
        logger.warning("No regularization strengths or solvers provided.")
        return None

    for solver in solvers:
        try:
            model = Pipeline([
                ('pca', PCA(n_components=n_components, random_state=random_state)),
                ('log_reg', LogisticRegressionCV(
                    Cs=Cs,
                    cv=cv or 5,
                    solver=solver,
                    random_state=random_state,
                    max_iter=max_iter,
                    scoring='accuracy'
                ))
            ])

            model.fit(data.X_train, data.y_train)

            log_reg = model.named_steps['log_reg']
            current_score = model.score(data.X_test, data.y_test)  # accuracy

            if current_score > best_score:
                best_score = current_score
                best_params.update({
                    "C": log_reg.C_[0] if hasattr(log_reg.C_, '__len__') else log_reg.C_,
                    "coef": log_reg.coef_.copy() if log_reg.coef_ is not None else None,
                    "intercept": (
                        log_reg.intercept_[0]
                        if hasattr(log_reg.intercept_, '__len__')
                        else log_reg.intercept_
                    ),
                    "score": best_score,
                    "solver": solver
                })

        except Exception as e:
            logger.exception(f"Failed to evaluate solver '{solver}': {str(e)}")
            continue

    return best_params if best_score != float('-inf') else None


In [52]:
def pca_gridcv_svm(
    data,
    cs: Sequence[float],
    kernel: Union[str, Sequence[str]] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    n_components: Union[float, int, str] = 0.95,
    **args
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """
    Performs PCA + SVM Grid Search with cross-validation.

    Args:
        data: Training data object with attributes X_train and y_train.
        cs: List of C values to test; all must be positive.
        kernel: Kernel type(s) to try (e.g., 'linear', 'rbf', etc.)
        gammas: Gamma values to try or 'auto'.
        cv: Number of cross-validation folds (default=5).
        n_components: Number of components for PCA (can be int, float, or 'mle').

    Returns:
        Dictionary containing best parameters and evaluation metrics,
        or None if an error occurred during fitting.
    """
    # --- 输入验证 ---
    if not cs:
        raise ValueError("Parameter 'cs' must contain at least one value.")

    if cv is not None and cv <= 0:
        raise ValueError("Parameter 'cv' must be a positive integer.")

    if any(c <= 0 for c in cs):
        raise ValueError("All values in 'cs' must be positive.")

    # Validate data structure
    try:
        X_train, y_train = data.X_train, data.y_train
    except Exception as e:
        logger.error(f"Invalid training data provided: {e}")
        return None

    # --- 初始化参数网格 ---
    kernel_list = [kernel] if isinstance(kernel, str) else list(kernel)
    gamma_list = [gammas] if isinstance(gammas, str) else list(gammas)
    c_list = list(cs)

    param_grid = {
        "GridCV_svm__C": c_list,
        "GridCV_svm__kernel": kernel_list,
        "GridCV_svm__gamma": gamma_list
    }

    # --- 构建 Pipeline ---
    pipeline = Pipeline([
        ('pca', PCA(n_components=n_components)),
        ('GridCV_svm', GridSearchCV(
            estimator=SVC(random_state=42),
            param_grid=param_grid,
            cv=cv,
            n_jobs=-1,
            verbose=0,
            scoring='accuracy'
        ))
    ])

    # --- 执行拟合并获取结果 ---
    try:
        pipeline.fit(X_train, y_train)

        # 如果没有找到有效的模型（例如因为所有组合都失败）
        if pipeline.named_steps['GridCV_svm'].best_score_ == float('-inf'):
            logger.warning("No valid model found during grid search.")
            return None

        # 提取最佳参数及支持向量
        best_params = pipeline.named_steps['GridCV_svm'].best_params_.copy()
        best_score = pipeline.named_steps['GridCV_svm'].best_score_

        # 添加额外信息
        best_params["score"] = best_score
        best_params["support_vectors"] = pipeline.named_steps['GridCV_svm'].best_estimator_.support_vectors_

        return best_params

    except Exception as e:
        logger.exception(f"An error occurred during grid search: {e}")
        return None

### 手动训练

In [ ]:
lr_solver:Sequence[SolverType]=["lbfgs"]
logger.debug(logistic_regression(data,Cs=[1,10,100],cv=5,solvers=lr_solver,max_iter=10000))

In [ ]:
logger.debug(pca_logistic_regression(data,Cs=[10,100],cv=5,solvers=lr_solver))

In [ ]:
logger.debug(randomCV_svm(data,Cs=[10,100]))

In [ ]:
logger.debug(GridCV_svm(data,Cs=[10,100]))

## agent workflow

In [ ]:
# 1. 定义 Pydantic 模型
class Action(BaseModel):
    action_type: Literal["calculate", "get_weather"]
    arguments: dict = Field(description="执行动作所需的参数")

class Plan(BaseModel):
    plan_summary: str = Field(description="计划的简短摘要")
    actions: List[Action] = Field(description="需要执行的行动列表")

# 2. 定义 LangGraph State
class State(BaseModel):
    user_query: str
    plan: Union[Plan, None] = None
    results: List[str] = Field(default_factory=list)
    messages: Annotated[list, add_messages]

### 工具类

In [53]:
@tool
async def get_logistic_regression(
    params: Dict[str, Union[float, str, Sequence[float], None]]
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """选择逻辑回归训练器
    
    Args:
        params: 包含训练参数的字典，支持以下键：
            - 'Cs': 正则化强度列表
            - 'cv': 交叉验证折数
            - 'solvers': 求解器列表
            - 其他参数将传递给逻辑回归模型
            
    Returns:
        训练结果字典，包含模型性能指标和参数信息
    """
    # 参数验证和默认值设置
    cs = params.get('Cs', [0.1, 1.0, 10.0])  # 默认正则化强度
    cv = params.get('cv', 5)  # 默认5折交叉验证
    solvers = params.get('solvers', ['lbfgs'])  # 默认求解器
    
    # 验证输入参数的有效性
    if not isinstance(cs, (list, tuple)):
        raise ValueError("Cs must be a list or tuple of regularization strengths")
    
    if not isinstance(cv, int) or cv <= 0:
        raise ValueError("cv must be a positive integer")
    
    if not isinstance(solvers, (list, tuple)):
        raise ValueError("solvers must be a list or tuple of solver names")
    
    # 从参数中提取其他可能的参数
    other_params = {k: v for k, v in params.items() 
                   if k not in ['Cs', 'cv', 'solvers']}
    
    try:
        # 根据是否启用PCA选择不同的训练方法
        if params.get('enable_pca', True):
            return await pca_logistic_regression(
                data, cs, cv, solvers, **other_params
            )
        else:
            return await logistic_regression(
                data, cs, cv, solvers, **other_params
            )
    except Exception as e:
        # 添加错误处理以提高健壮性
        raise RuntimeError(f"Logistic regression training failed: {str(e)}")

In [55]:
@tool
async def get_svm(
    params: Dict[str, Union[float, str, Sequence[float], None]]
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """选择 SVM 训练器

    Args:
        params: 包含训练参数的字典，支持以下键：
            - 'Cs': 正则化强度列表
            - 'cv': 交叉验证折数
            - 'solvers': 求解器列表
            - 'enable_pca': 是否启用 PCA，默认为 True
            - 其他参数将传递给 SVM 模型

    Returns:
        训练结果字典，包含模型性能指标和参数信息

    Raises:
        ValueError: 当输入参数无效时抛出
        RuntimeError: 当训练过程失败时抛出
    """
    # 参数验证与默认值设置
    cs = params.get('Cs', [0.1, 1.0, 10.0])  # 默认正则化强度
    cv = params.get('cv', 5)  # 默认5折交叉验证
    solvers = params.get('solvers', ['lbfgs'])  # 默认求解器
    enable_pca = params.get('enable_pca', True)  # 是否启用 PCA

    # 输入参数校验
    if not isinstance(cs, (list, tuple)):
        raise ValueError("Cs must be a list or tuple of regularization strengths")

    if not isinstance(cv, int) or cv <= 0:
        raise ValueError("cv must be a positive integer")

    if not isinstance(solvers, (list, tuple)):
        raise ValueError("solvers must be a list or tuple of solver names")

    # 提取其余参数
    other_params = {
        k: v for k, v in params.items()
        if k not in ['Cs', 'cv', 'solvers', 'enable_pca']
    }

    try:
        # 根据是否启用 PCA 调用不同训练方法
        if enable_pca:
            return await pca_gridcv_svm(data, cs, cv, solvers, **other_params)
        else:
            return await gridcv_svm(data, cs, cv, solvers, **other_params)
    except Exception as e:
        # 更详细的错误信息以便调试
        raise RuntimeError(f"SVM training failed: {str(e)}") from e

In [ ]:
llm = ChatOpenAI(
        model=cfg.openai_model, 
        api_key=cfg.openai_api_key,
        base_url=cfg.openai_base_url,
        temperature=cfg.temperature,
        max_retries=cfg.max_retries,
        timeout=cfg.request_timeout,
        streaming=True,
        stream_usage=True,
        extra_body={
            "enable_thinking":cfg.enable_thinking
        },
    )
structured_llm = llm.with_structured_output(Plan)

In [62]:
async def plan_node(state: State) -> dict:
    """规划节点：根据用户请求生成结构化计划"""
    logger.info(f"开始规划，用户请求: {state.user_query}")
    try:
        # 要求 LLM 制定行动计划
        plan: Plan = await structured_llm.ainvoke(
            f"用户请求: {state.user_query}. 请为此请求制定一个详细的行动计划。"
        )
        logger.info(f"规划成功，摘要: {plan.plan_summary}")
        # 记录生成的完整 Plan
        logger.debug(f"完整计划: {plan.model_dump_json(indent=2)}")
        return {"plan": plan}
    except Exception as e:
        logger.error(f"规划失败: {e}", exc_info=True)
        # 返回一个空计划并附带错误消息，避免流程中断
        return {
            "plan": None,
            "messages": [AIMessage(content=f"规划步骤时出错: {str(e)}")]
        }

In [63]:
tool_node = ToolNode([get_logistic_regression,get_svm])
async def execute_tools_node(state: State) -> dict:
    """执行节点：将 Plan 中的行动转为 ToolNode 调用并收集结果"""
    if not state.plan or not state.plan.actions:
        logger.warning("没有需要执行的行动")
        return {"results": [], "messages": []}

    logger.info(f"准备执行 {len(state.plan.actions)} 个行动")

    # 1. 构造 tool_calls 列表（符合 OpenAI function call 格式）
    tool_calls = []
    for i, action in enumerate(state.plan.actions):
        # Pydantic 的字典可通过 model_dump 得到，但 arguments 已经是 dict
        call = {
            "id": f"call_{i}",
            "function": {
                "name": action.action_type,
                "arguments": action.arguments,  # 这里应当与工具函数的参数一致
            },
            "type": "tool_call",
        }
        tool_calls.append(call)

    ai_message = AIMessage(content="", tool_calls=tool_calls)

    try:
        # 2. 异步调用 ToolNode，自动并发执行所有工具
        #    设置超时防止某个工具卡死
        tool_results = await asyncio.wait_for(
            tool_node.ainvoke({"messages": [ai_message]}),
            timeout=30.0  # 可根据实际情况调整
        )
    except asyncio.TimeoutError:
        logger.error("工具执行整体超时")
        results = ["部分工具执行超时"]
    except Exception as e:
        logger.error(f"工具执行异常: {e}", exc_info=True)
        results = [f"工具执行失败: {str(e)}"]
    else:
        # ToolNode 返回的 messages 是 ToolMessage 列表
        results = [msg.content for msg in tool_results["messages"]]
        logger.info(f"所有工具执行完毕，结果: {results}")

    return {"results": results}

In [65]:
async def respond_node(state: State) -> dict:
    """响应节点：根据原始请求、计划和执行结果生成自然语言回复"""
    logger.info("生成最终回复")

    # 如果没有工具执行结果，给出默认回复
    if not state.results:
        default_msg = "抱歉，没有找到可执行的操作。"
        logger.warning(default_msg)
        return {"messages": [AIMessage(content=default_msg)]}

    # 构造提示，让 LLM 整合信息
    context = f"用户问题：{state.user_query}\n"
    context += f"计划摘要：{state.plan.plan_summary if state.plan else '无'}\n"
    results_formatted = "\n".join(f"- {r}" for r in state.results)
    context += f"执行结果：\n{results_formatted}\n"
    context += "请根据以上信息，生成一段友好、清晰的回复给用户（使用中文）。"

    try:
        response = await llm.ainvoke(context)
        msg_content = response.content if hasattr(response, 'content') else str(response)
        logger.info(f"回复生成成功: {msg_content}")
    except Exception as e:
        logger.error(f"生成回复失败: {e}", exc_info=True)
        msg_content = "处理请求时出现错误，请稍后重试。"

    return {"messages": [AIMessage(content=msg_content)]}

In [66]:
def should_execute_tools(state: State) -> Literal["execute_tools", "respond"]:
    """路由函数：判断是否还有需要执行的工具"""
    if state.plan and state.plan.actions:
        return "execute_tools"
    return "respond"

def create_graph() -> StateGraph:
    """构建并编译 LangGraph 工作流"""
    workflow = StateGraph(State)

    # 添加节点
    workflow.add_node("planner", plan_node)
    workflow.add_node("executor", execute_tools_node)
    workflow.add_node("responder", respond_node)

    # 定义边
    workflow.add_edge(START, "planner")
    workflow.add_conditional_edges(
        "planner",
        should_execute_tools,
        {
            "execute_tools": "executor",
            "respond": "responder",
        },
    )
    workflow.add_edge("executor", "responder")
    workflow.add_edge("responder", END)

    return workflow.compile()

In [67]:
async def main():
    logger.info("启动 LangGraph 结构化代理")

    # 确保 API Key 已设置
    if not cfg.openai_api_key:
        raise ValueError("OPENAI_API_KEY 未设置，请在环境变量或 .env 文件中配置")

    graph = create_graph()
    user_query = "请帮我找出最佳算法及其最佳参数和准确率"

    initial_state = State(user_query=user_query, messages=[])

    # 流式执行，收集最终状态
    final_state = None
    async for state in graph.astream(initial_state, stream_mode="values"):
        final_state = state
        logger.debug("当前状态:{}",state)

    if final_state and final_state.messages:
        print("\n🤖 最终回复：")
        print(final_state.messages[-1].content)
    else:
        print("未生成回复。")


In [68]:
asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop